# Project pipeline — Northern Bahr el Ghazal flood forecasting (JBG060 / ZHL)

Run the notebook top-to-bottom from the repository root. Model selection between the grid cells is done manually by comparing the `cv_metrics_summary_aweil.csv` tables in the respective `LightGBM_v1/outputs/outputs_*/tables/` folders.


## 1. Pre-processing

The model consumes:

| Input | Built by | Used for |
|---|---|---|
| `EDA_flood_masks/outputs/tables/weekly_county_floods.csv` | flood-label cell (this section) | flood labels |
| `EDA_hydrometeorology/outputs_county/tables/aweil_monthly_agricultural_exposure.csv` | exposure cell (this section) | exposure context |
| `raw_data/Administrative boundaries/ssd_admin2.geojson` | raw data | county boundaries |
| `raw_data/rainfall and runoff`, `raw_data/Dartmouth Flood Observatory`, `raw_data/Water levels lakes` | raw data | precipitation/runoff, river gauge, lake level |
| `processing_data/evapotranspiration/ET_<year>_9.475N_30.725E_processed.csv` | `process_et_NBeG.py` (this section) | ET0 features |

Three cells below produce the two derived tables and the ET0 CSVs:

1. `process_et_NBeG.py` processes the raw AgERA5 ET NetCDFs into annual CSVs.
2. The flood-label cell re-aggregates the raw flood-mask Parquets into the weekly county table, using the exact `EDA_flood_masks` EDA chain.
3. The exposure cell samples the 2024 flood detections against the ASAP crop and rangeland masks, using the exact `EDA_hydrometeorology` EDA chain.

Missing raw inputs raise an error; nothing is substituted.


In [1]:
import sys

%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

from LightGBM_v1 import config

# Check the raw inputs up front; missing sources raise an error later
# in the pipeline.
raw_inputs = {
    "admin-2 boundaries": config.BOUNDARY_ADMIN2,
    "raw flood-mask Parquets": config.RAW_DATA / "flood_masks",
    "ASAP crop and rangeland masks": config.RAW_DATA / "farmland",
    "raw ERA5 rainfall/runoff": config.RAW_DATA / "rainfall and runoff",
    "Dartmouth gauge": config.GAUGE_ROOT,
    "Lake Albert levels": config.LAKE_ROOT,
    "raw AgERA5 ET NetCDFs": config.RAW_DATA / "evapotranspiration",
}
print("External raw inputs:")
for label, path in raw_inputs.items():
    print(f"  [{'OK' if Path(path).exists() else 'MISSING':>7}] {label}: {path}")

# Derived artifacts regenerated by the pre-processing cells below.
generated = {
    "weekly flood labels (flood-label cell)": config.AWEIL_WEEKLY_FLOODS_CSV,
    "monthly agricultural exposure (exposure cell)": config.AWEIL_MONTHLY_EXPOSURE_CSV,
}
print("\nRegenerated by the pre-processing cells below:")
for label, path in generated.items():
    print(f"  [{'OK' if Path(path).exists() else 'MISSING':>7}] {label}: {path}")

n_et = len(list(Path(config.ET0_ROOT).glob("ET_*_processed.csv")))
missing_et = len(config.YEARS) - n_et
et_state = "OK" if missing_et == 0 else "MISSING"
print(f"  [{et_state:>7}] processed ET0 CSVs: {n_et}/{len(config.YEARS)} years in {config.ET0_ROOT}")

todo = [label for label, path in generated.items() if not Path(path).exists()]
if missing_et:
    todo.append(f"{missing_et} ET0 year(s)")
if todo:
    print("-> run the pre-processing cells below for: " + "; ".join(todo))

External raw inputs:
  [     OK] admin-2 boundaries: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Administrative boundaries/ssd_admin2.geojson
  [     OK] raw flood-mask Parquets: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/flood_masks
  [     OK] ASAP crop and rangeland masks: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/farmland
  [     OK] raw ERA5 rainfall/runoff: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/rainfall and runoff
  [     OK] Dartmouth gauge: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Darthmouth Flood Observatory
  [     OK] Lake Albert levels: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/Water levels lakes
  [     OK] raw AgERA5 ET NetCDFs: /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/raw_data/evapotranspiration

Regenerated by the pre-processing cells below:
  [     OK] weekly flood labels (flood-label cell): /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/EDA_flood_masks/outputs/t

In [3]:
# Process the raw AgERA5 ET NetCDFs into the annual CSVs.
!{sys.executable} process_et_NBeG.py

PROCESS ET — NORTHERN BAHR EL GHAZAL — 9.475N 30.725E — 2000-2025
Processing ET:   0%|                                    | 0/26 [00:00<?, ?it/s].nc file for year 2000 already processed in processing_data/evapotranspiration
.nc file for year 2001 already processed in processing_data/evapotranspiration
.nc file for year 2002 already processed in processing_data/evapotranspiration
.nc file for year 2003 already processed in processing_data/evapotranspiration
.nc file for year 2004 already processed in processing_data/evapotranspiration
.nc file for year 2005 already processed in processing_data/evapotranspiration
.nc file for year 2006 already processed in processing_data/evapotranspiration
.nc file for year 2007 already processed in processing_data/evapotranspiration
.nc file for year 2008 already processed in processing_data/evapotranspiration
.nc file for year 2009 already processed in processing_data/evapotranspiration
.nc file for year 2010 already processed in processing_data/evapo

In [4]:
# Rebuild the weekly flood-label table from the raw flood-mask Parquets.
# Missing raw files raise an error; nothing is substituted.
from EDA_flood_masks.flood_eda import (
    load_flood_observations,
    load_study_counties,
    summarise_weekly,
    validate_outputs,
)

flood_root = config.RAW_DATA / "flood_masks"
label_counties = load_study_counties(config.BOUNDARY_ADMIN2)
label_counties = label_counties.sort_values("county").reset_index(drop=True)
label_observations = load_flood_observations(flood_root, label_counties)
weekly_labels = summarise_weekly(label_observations)
validate_outputs(label_observations, weekly_labels)

config.AWEIL_WEEKLY_FLOODS_CSV.parent.mkdir(parents=True, exist_ok=True)
weekly_labels.to_csv(config.AWEIL_WEEKLY_FLOODS_CSV, index=False)
print(f"{len(weekly_labels):,} county-weeks with detections -> {config.AWEIL_WEEKLY_FLOODS_CSV}")
print(
    f"Detections span {label_observations['date'].min():%Y-%m-%d} "
    f"to {label_observations['date'].max():%Y-%m-%d}"
)

1,457 county-weeks with detections -> /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/EDA_flood_masks/outputs/tables/weekly_county_floods.csv
Detections span 2000-10-21 to 2025-12-21


In [5]:
# Rebuild the monthly agricultural exposure table
# (2024 flood detections x ASAP crop and rangeland masks).
# Missing raw files raise an error; nothing is substituted.
from pathlib import Path

import numpy as np

from EDA_hydrometeorology.hydrological_analysis import (
    agricultural_baseline,
    load_aweil_counties,
    load_aweil_flood_observations,
    seasonal_agricultural_exposure,
)

flood_root = config.RAW_DATA / "flood_masks"
mask_paths = {
    "crop": str(config.RAW_DATA / "farmland" / "asap_mask_crop_v04.tif"),
    "rangeland": str(config.RAW_DATA / "farmland" / "asap_mask_rangeland_v04.tif"),
}
for land_type, path in mask_paths.items():
    if not Path(path).is_file():
        raise RuntimeError(f"Missing {land_type} exposure raster: {path}")

exposure_counties = load_aweil_counties(config.BOUNDARY_ADMIN2)
exposure_observations = load_aweil_flood_observations(
    flood_root, exposure_counties, years=np.array([2024])
)
exposure_baseline = agricultural_baseline(exposure_counties, mask_paths)
exposure = seasonal_agricultural_exposure(exposure_observations, exposure_counties, mask_paths)
exposure = exposure.merge(exposure_baseline, on="county", how="left")
exposure["crop_exposed_percent"] = 100 * exposure["crop_exposed_hectares"] / exposure["crop_hectares"]
exposure["rangeland_exposed_percent"] = 100 * exposure["rangeland_exposed_hectares"] / exposure["rangeland_hectares"]

config.AWEIL_MONTHLY_EXPOSURE_CSV.parent.mkdir(parents=True, exist_ok=True)
exposure.to_csv(config.AWEIL_MONTHLY_EXPOSURE_CSV, index=False)
print(f"{len(exposure)} county x month x flood-type rows -> {config.AWEIL_MONTHLY_EXPOSURE_CSV}")

42 county x month x flood-type rows -> /home/tijnq/documents/uni/jbg060/JBG060_ZHL_2026/EDA_hydrometeorology/outputs_county/tables/aweil_monthly_agricultural_exposure.csv


## 2. Level 1 — TabPFN zero-shot reference

Zero-shot TabPFN classifiers/regressors: they never train on our data, so this shows how much signal the feature matrix carries without any tuning.

- Outputs → `TabPFN_v1/outputs/outputs_v3/tables/`.
- Hyperparameters are not tunable; `--params` has no effect here.


In [ ]:
from LightGBM_v1.run import main

# Zero-shot TabPFN reference (outputs -> TabPFN_v1/outputs/outputs_v3/tables/).
main("tabpfn")

## 3. Level 1 — baseline features, baseline hyperparameters

Trains LightGBM with the baseline feature set and baseline hyperparameters. This is the reference run the later sections compare against.

- Early stopping on each fold's validation year; the boundary week is purged per fold.
- The climatology and Markov-persistence references are re-estimated inside each fold's training window, so they never see validation/test weeks.
- Outputs → `LightGBM_v1/outputs/outputs_baseline/tables/`.


In [ ]:
from LightGBM_v1.run import main


# Baseline features and hyperparameters
# (outputs -> LightGBM_v1/outputs/outputs_baseline/tables/).
main("lightgbm_baseline")

## 4. Level 1 — Optuna tuning on the baseline features

Tunes the baseline model with Optuna: three resumable single-head studies that share one feature matrix (trial #0 of each re-enqueues today's defaults as an anchor).

| Stage | Study | Head | Objective |
|---|---|---|---|
| 1 | `lgbm_aweil_baseline_v3_brier` | det | Brier |
| 2 | `lgbm_aweil_baseline_v3_pinball` | quantiles | mean pinball loss |
| 3 | `lgbm_aweil_baseline_v3_dur` | duration | per-horizon Brier |

50 trials per stage, 3600 s timeout per stage. Studies live in the SQLite store `LightGBM_v1/outputs/tuning/studies.db`, so re-running this cell resumes them. The winners are merged into `outputs/tuning/best_params_lgbm_aweil_baseline_v3_all_heads.json`; with `evaluate=True` the merged winners are also refit and scored on the frozen test years (`cv_metrics_tuned_lgbm_aweil_baseline_v3_all_heads.csv`; the tuned duration metrics come from the full pipeline run in section 5).

`run.py` does not read the JSON at run time — `params_name="baseline_optuna"` uses the values hard-coded in `LightGBM_v1/lgbm_model.py::PARAM_SETS`. After tuning, copy the `det` / `quantiles` / `duration` winners from the JSON into `PARAM_SETS["baseline_optuna"]` before running section 5.


In [ ]:
from LightGBM_v1.tuning import tune_lightgbm

study_baseline = tune_lightgbm(
    feature_set="baseline",
    objective="all_heads",  # det (Brier) + quantiles (pinball) + duration (Brier)
    n_trials=50,            # per stage
    evaluate=True,          # also refit the merged winners on the frozen test years
)

## 5. Level 1 — baseline features with the Optuna winners

Re-runs the section-3 protocol with the tuned per-head parameters. Outputs go to `LightGBM_v1/outputs/outputs_baseline_optuna/tables/` so the baseline outputs stay untouched and comparable. As-is, this cell reproduces the completed tuned run.


In [ ]:
from LightGBM_v1.run import main

# Baseline features with the tuned winners from PARAM_SETS['baseline_optuna']
# (outputs -> LightGBM_v1/outputs/outputs_baseline_optuna/tables/).
main("lightgbm_baseline", "baseline_optuna")

## 6. Level 1 — engineered features, baseline defaults

Tests whether two additions improve on the baseline. Starting from the frozen baseline matrix it adds `cnty_tp_w7` / `cnty_ro_w7` (7-day ERA5 precipitation/runoff rolling sums over each county's own bounding box) and the `county` identity as a categorical feature, and drops `year` (the cyclic `month` / `week_of_year` encodings stay) → 52 columns. Same folds, embargo and frozen baseline hyperparameters (`PARAM_SETS["fe_baseline"]` = no overrides). Outputs → `LightGBM_v1/outputs/outputs_fe/tables/`.


In [ ]:
from LightGBM_v1.run import main

# Engineered feature set with the frozen baseline defaults
# (outputs -> LightGBM_v1/outputs/outputs_fe/tables/).
main("lightgbm_fe", "fe_baseline")

## 7. Level 1 — Optuna tuning on the engineered features

Same three-stage all-heads design as section 4, on the 52-column engineered matrix: studies `lgbm_aweil_fe_v3_{brier,pinball,dur}` in `outputs/tuning/studies.db`, merged winners in `outputs/tuning/best_params_lgbm_aweil_fe_v3_all_heads.json`, and (with `evaluate=True`) tuned test-year metrics in `cv_metrics_tuned_lgbm_aweil_fe_v3_all_heads.csv`.

After tuning, copy the JSON's `det` / `quantiles` / `duration` groups into `PARAM_SETS["fe_optuna"]` in `LightGBM_v1/lgbm_model.py` before running section 8.


In [ ]:
from LightGBM_v1.tuning import tune_lightgbm

study_fe = tune_lightgbm(
    feature_set="fe",
    objective="all_heads",  # det (Brier) + quantiles (pinball) + duration (Brier)
    n_trials=50,            # per stage
    evaluate=True,          # also refit the merged winners on the frozen test years
)

## 8. Level 1 — engineered features with the Optuna winners

Re-runs the section-6 protocol with the tuned parameters. Outputs → `LightGBM_v1/outputs/outputs_fe_optuna/tables/`.


In [ ]:
from LightGBM_v1.run import main

# Engineered features with the tuned winners from PARAM_SETS['fe_optuna']
# (outputs -> LightGBM_v1/outputs/outputs_fe_optuna/tables/).
main("lightgbm_fe", "fe_optuna")

## 9. Level 1 — final fit mode

`final_fit("lightgbm_fe", "fe_baseline")` retrains the chosen configuration once — the CV protocol's expanding-window logic taken one fold further: train on 2000–2024, early-stop on 2025. It then forecasts the latest feature-complete week (2025-12-29, with inputs ending Dec 2025): `det_prob` and `q10/q50/q90` for that week, plus `det_prob_h1/h2` from the duration heads (h3–h4 are not reported because they lose to climatology and persistence in `duration_metrics_aweil.csv`).

Later weeks are deliberately not forecast: they would need weather data beyond Dec 2025, and partial rolling windows would silently change what the features mean — the completeness gate in `run.py` drops such rows. Once 2026 ERA5/ET0 and flood labels are added, this cell forecasts the then-current week.

Output: `LightGBM_v1/outputs/outputs_final_fit/tables/final_forecast_aweil.csv` (columns `county, week, det_prob, q10, q50, q90, det_prob_h1, det_prob_h2`), which feeds the section-11 maps.


In [ ]:
from LightGBM_v1.run import final_fit

# Retrain lightgbm_fe (baseline defaults) on 2000-2024 with early stopping on
# 2025 and forecast the latest feature-complete week (2025-12-29) plus
# horizons h1-h2
# (output -> LightGBM_v1/outputs/outputs_final_fit/tables/final_forecast_aweil.csv).
final_fit("lightgbm_fe", "fe_baseline")


LIGHTGBM_FE FINAL FIT — HYPERPARAMETERS: fe_baseline

[1/4] Building weekly feature matrix with Friday 3-day embargo...
Daily total precipitation and runoff from ERA5 reanalysis loaded from dates 2000-01-01 to 2025-12-31
<xarray.Dataset> Size: 628MB
Dimensions:     (valid_time: 9497, latitude: 145, longitude: 57)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 76kB 2000-01-01 ... 2025-12-31
  * latitude    (latitude) float64 1kB 33.0 32.75 32.5 32.25 ... -2.5 -2.75 -3.0
  * longitude   (longitude) float64 456B 23.0 23.25 23.5 ... 36.5 36.75 37.0
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 314MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
    ro          (valid_time, latitude, longitude) float32 314MB dask.array<chunksize=(1, 145, 57), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Con

## 10. Level 2 — training-validation (pixel scale)

Task B currently uses the pixel-climatology spatial prior with an honest evaluation; a trained per-cell LightGBM or U-Net is plan-only (`climatology_maps/UNET_PLAN.md`).

1. `pixel_climatology.py` — per ~232 m grid cell and calendar month, from the raw flood-mask Parquets (2000–2025): flood frequency, frequency given the county flooded (the signature used to spread a county flood across its cells), and the county-month probability `(k+0.5)/(n+1)`. Cached in `climatology_maps/outputs/climatology_data/`.
2. `evaluate_pixel_maps.py` — scores the LightGBM × pixel-climatology maps on the out-of-fold test weeks only (per-fold climatology trained on 2000–2008 / 2000–2014 / 2000–2020; 2025 excluded): pixel-week Brier/BSS vs climatology and persistence, AUROC/PR-AUC, threshold metrics (CSI/FAR/POD/…), weekly flooded-area MAE/bias and reliability. Written to `climatology_maps/outputs/forecast_maps/PIXEL_PERFORMANCE.md`.

Predictions default to the first committed LightGBM table found among `outputs_fe` → `outputs_fe_optuna` → `outputs_baseline_optuna` → `outputs_baseline` → `outputs_v3` (`cv_test_predictions_aweil.csv`); pass `--predictions <path>` to score a specific run's table instead.


In [ ]:
import sys

# One-time build of the per-pixel climatology cache (~30 s; needs the raw
# flood-mask Parquets). Add --rebuild to force a fresh build.
!{sys.executable} -m climatology_maps.pixel_climatology

In [ ]:
import sys

# Honest pixel-level validation of the LightGBM x pixel-climatology maps on the
# OOF test weeks (per-fold trained climatology; 2025 excluded).
!{sys.executable} -m climatology_maps.evaluate_pixel_maps

## 11. Level 2 — final fit mode

`climatology_maps.forecast_maps --mode final` downscales the section-9 final-fit forecast (`outputs_final_fit/tables/final_forecast_aweil.csv`: county `det_prob` and `q10/q50/q90` for 2025-12-29, plus `det_prob_h1/h2`) onto the ~232 m pixel grid with the per-pixel climatology, reusing the compositing and rendering code validated in section 10. Outputs land in `climatology_maps/outputs/outputs_final_fit/`: `prob/sev/worst/clim_<week>.png`, an interactive HTML and `week_summary.csv`.

The severity layers spread the county's `q50` predicted area with the flood-frequency-given-flood signature, so months with no historical floods fall back to the annual pattern with severity 0 (the `prob` layer still places the county forecast). `--horizon 1|2` re-renders the map for the duration heads; h3/h4 are not in the final table because they lose to climatology and persistence. A trained per-cell Level-2 model would replace the climatology spreading — plan-only (`climatology_maps/UNET_PLAN.md`).


In [ ]:
import os
import sys

# The maps modules resolve raw data and outputs relative to the repo root, so
# make sure we run from there even if the kernel was started elsewhere.
for _ in range(4):
    if os.path.isdir("climatology_maps"):
        break
    os.chdir("..")
assert os.path.isdir("climatology_maps"), "run main.ipynb from inside the project repo"

# Downscale the section-9 final-fit forecast onto the pixel grid with the
# per-pixel climatology (outputs -> climatology_maps/outputs/outputs_final_fit/).
# Add --horizon 2 for the +2-weeks view, --metric prob for the probability layer.
!{sys.executable} -m climatology_maps.forecast_maps --mode final

## 12. Forecast-week risk map (after each model run)

The model scores whole counties per week ("Aweil East has a 21% chance of flood water this week"). This section maps what that means on the ground: two figures (cattle, crop), each with three panels — the Level-2 flood probability (left), the asset map (middle), and their product (right): per pixel, chance of water × what is standing there.

Four inputs:

1. This week's model forecast — `LightGBM_v1/outputs/outputs_final_fit/tables/final_forecast_aweil.csv`: `det_prob` for this week and `det_prob_h1`/`det_prob_h2` for the two following weeks.
2. Where water goes inside a county — the model only scores counties, so this comes from the 2000–2025 satellite record: in past weeks when a county was flooded, which pixels were wet. The cell below loads it with Level 2's `get_climatology`, and the module downscales the county chance with Level 2's `compose_week` (the same function used in section 11).
3. Cattle — `raw_data/farmland/geonode__cattle_gha.tif`: head per km².
4. Crop land — `raw_data/farmland/asap_mask_crop_v04.tif`: share of each ~0.5 km cell that is crop land.

The rule, for both assets: expected exposure = chance of water × how much of the asset is there.

- cattle: chance × head per km² × pixel area → expected head
- crop: a ~0.5 km crop cell counts once when any of its satellite pixels is wet → expected hectares

The same product from the shell after any model run: `python -m risk_exposure.forecast_risk_map` (defaults to the final-fit forecast; `--predictions PATH` for another run's table, `--max-weeks N` to cap long tables).


In [ ]:
# Each figure shows the flood probability (left), the asset map (middle)
# and their product (right) for the forecast week; the 1- and 2-week
# persistence chances stay in the CSV. The climatology is loaded here and
# passed to Level 2's compose_week for the per-pixel probability. A missing
# forecast table or raster raises an error; nothing is substituted.
from IPython.display import Image, display

from climatology_maps.pixel_climatology import get_climatology
from risk_exposure.forecast_risk_map import (
    CLIMATOLOGY_CACHE,
    build_forecast_risk_maps,
)

climatology = get_climatology(cache_dir=CLIMATOLOGY_CACHE)

risk_summary, risk_figures = build_forecast_risk_maps(climatology=climatology)

for risk_figure in risk_figures:
    print(risk_figure)
    display(Image(filename=str(risk_figure)))

risk_summary


## 13. The same maps on a flood-season week (training runs)

Section 12 maps the operational December forecast, a dry week. To see the same machinery with real water, this section runs it on the training runs instead: the model's out-of-fold predictions for the weeks it was cross-validated on (`LightGBM_v1/outputs/outputs_fe/tables/cv_test_predictions_aweil.csv`). Same inputs, same rule, only the week changes — pinned to 2022-11-14, Aweil South's biggest flood in the record (92% Aweil South and Aweil East, 242 km² of observed water). As in section 12, the cell loads Level 2's climatology and passes it in, so the left panel is Level 2's output.

Caveats: these are held-out predictions about the past (a hindcast, not a forecast), and the cattle and crop rasters are static asset maps (cattle: 2010, crop: ~2020), not 2022's.


In [ ]:
# Section 12's code on a training-run week: the table is filtered to
# 2022-11-14 (Aweil South's biggest flood in the record) and the
# _training names keep section 12's outputs untouched. The climatology is
# loaded here as in section 12.
import pandas as pd
from IPython.display import Image, display

from climatology_maps.pixel_climatology import get_climatology
from risk_exposure.forecast_risk_map import (
    CLIMATOLOGY_CACHE,
    PROJECT_ROOT,
    TABLE_DIR,
    build_forecast_risk_maps,
)

WEEK = "2022-11-14"

training_cv = pd.read_csv(
    PROJECT_ROOT / "LightGBM_v1/outputs/outputs_fe/tables/cv_test_predictions_aweil.csv",
    parse_dates=["week"],
)
training_week_csv = TABLE_DIR / "forecast_risk_training_week.csv"
training_cv[training_cv["week"] == WEEK].to_csv(training_week_csv, index=False)

climatology = get_climatology(cache_dir=CLIMATOLOGY_CACHE)

training_summary, training_figures = build_forecast_risk_maps(
    predictions_path=training_week_csv,
    climatology=climatology,
    out_table=TABLE_DIR / "forecast_risk_summary_training.csv",
)

for training_figure in training_figures:
    print(training_figure)
    display(Image(filename=str(training_figure)))

training_summary
